# Mistral Model Cache Compression and Projection Analysis

This notebook is adapted from computing_proj.ipynb to work with Mistral models.
It demonstrates KV cache compression using SVD and other techniques on Mistral architectures.

In [ ]:
import torch
print(torch.__version__)  # Check PyTorch version
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Should return the number of GPUs
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # Should print the name of your GPU
    
from transformers import pipeline
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import os
from huggingface_hub import snapshot_download

# Import Mistral-specific implementations
from mistral_model_query import MistralModelQuery
from custom_cache_mistral import CustomCacheMistral

import numpy as np
from dataset_utils_mistral import prepare_mistral_dataset
from process_cache_mistral import computing_cache, compute_proj_SVD_mem, compute_proj_SVD_Eigen_mem, compute_proj_KQT_mem
from proj_utils import read_proj, read_spectrums, computing_rank
from evaluation_mistral import testing_proj_mistral

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Model and Tokenizer Setup

Load Mistral model and tokenizer. Note that Mistral models support longer sequences than Llama2.

In [ ]:
with open("../hf_token.txt") as f:
    hf_token = f.read().strip()


# Loading tokenizer - Update this path to your Mistral model


model_name = "mistralai/Mistral-7B-v0.3"
save_dir = None

snapshot_download(
    repo_id=model_name,
    cache_dir=save_dir,
    token=hf_token
    # use_auth_token=True  # only if model is gated/restricted
)

print("Mistral models typically support up to 32k tokens with sliding window attention")

## Model Loading

Load different variants of Mistral models:
- MistralModelQuery: stores query cache, needs CustomCacheMistral
- MistralModelQueryNoRoPE: stores query cache, no RoPE for keys and queries
- MistralModelNoRoPEinCache: no RoPE in the cache (keys cached without RoPE)

In [ ]:
# Load the Mistral model variant you want to work with
model_path = None

tokenizer = AutoTokenizer.from_pretrained(model_path,token=hf_token, torch_dtype=torch.float16)

# Option 1: Full Mistral model with query caching
model = MistralModelQuery.from_pretrained(model_path, device_map='auto', torch_dtype=torch.float16)


print(f"Model loaded: {type(model).__name__}")

## Model Configuration

Extract key hyperparameters for Mistral model analysis.

In [ ]:
# List of hyperparameters for Mistral
D = model.config.hidden_size  # full hidden dimension D=h*d
h = model.config.num_attention_heads  # number of attention heads (typically 32)
hkv = model.config.num_key_value_heads  # number of key/value heads for GQA (typically 8)
d = D // h  # head dimension (calculated for Mistral since it doesn't have head_dim attribute)
L = model.config.num_hidden_layers  # number of layers
sliding_window = getattr(model.config, 'sliding_window', None) or 4096  # sliding window size (default 4096)
print(f"Model Configuration:")
print(f"  Hidden size (D): {D}")
print(f"  Attention heads (h): {h}")
print(f"  Key-Value heads (hkv): {hkv}")
print(f"  Head dimension (d): {d}")
print(f"  Number of layers (L): {L}")
print(f"  Sliding window: {sliding_window}")
print(f"  GQA ratio: {h // hkv}:1")

## Dataset Preparation

Prepare C4 dataset for Mistral model analysis. We can use longer sequences than with Llama2.

In [ ]:
# Dataset preparation for Mistral
# Mistral can handle longer sequences efficiently due to sliding window attention

# Option 1: Standard length sequences
traindata, valdata, _ = prepare_mistral_dataset(
    dataset_name='c4',
    model_name=model_path,
    nsamples_train=128,
    nsamples_test=32,
    seed=0,
    seqlen=2048
)

print(f"Training sequences: {len(traindata)}")
print(f"Validation sequences: {len(valdata)}")
print(f"Sequence length: {traindata[0].shape[1]}")

## Cache Computation

Compute and store the KQV cache for all training sequences.

In [ ]:
# Compute cache for Mistral model
cache_name = f"Mistral-7B_C4_train_{len(traindata)}_{traindata[0].shape[1]}_sliding"

computing_cache(model, cache_name, traindata, len(traindata))

print(f"Cache computation completed for {cache_name}")

## SVD Projection Computation

Compute SVD-based projections for cache compression.

In [9]:
proj_name = f"Mistral-7B_projSVD_C4_train_{len(traindata)}_{traindata[0].shape[1]}_sliding"

In [ ]:
# Compute SVD projections


compute_proj_SVD_mem(model, cache_name, len(traindata), proj_name, save=True)

print(f"SVD projection computation completed: {proj_name}")

In [11]:
# Clear GPU memory
torch.cuda.empty_cache()

## Eigendecomposition-based Projections

Compute eigendecomposition-based projections for comparison.

In [12]:
eigen_proj_name = f"Mistral-7B_projSVDEigen_C4_train_{len(traindata)}_{traindata[0].shape[1]}_sliding"

In [ ]:
# Compute eigendecomposition projections

compute_proj_SVD_Eigen_mem(model, cache_name, len(traindata), eigen_proj_name, save=True)

print(f"Eigen projection computation completed: {eigen_proj_name}")

In [14]:
# Clear GPU memory
torch.cuda.empty_cache()

## KQT Projection Computation

Compute projections based on K*Q^T analysis.

In [15]:
kqt_proj_name = f"Mistral-7B_projSVDkqt_C4_train_{len(traindata)}_{traindata[0].shape[1]}_sliding"

In [ ]:
# Compute eigendecomposition projections

compute_proj_KQT_mem(model, cache_name, len(traindata), kqt_proj_name, save=True)

print(f"KQT projection computation completed: {kqt_proj_name}")

In [17]:
# Clear GPU memory
torch.cuda.empty_cache()

## Spectrum Analysis

Analyze the singular value spectra to determine optimal compression ranks.

In [ ]:
# Read and analyze spectra
spec = read_spectrums(model, proj_name)

print("Spectrum analysis completed")
print(f"Spectrum shape: {len(spec)} layers")

## Rank Computation

Compute optimal ranks based on error tolerance.

In [ ]:
# Compute ranks for different error tolerances
eps = 0.1  # Error tolerance
ranks = computing_rank(model, spec, [[eps, eps] for _ in range(L)], square=True)

print(f"Computed ranks for eps={eps}:")
print(ranks)

# Calculate compression ratios
kq_compression = [d / rank[0] for rank in ranks]
vw_compression = [d / rank[1] for rank in ranks]

print(f"\nAverage K/Q compression ratio: {np.mean(kq_compression):.2f}x")
print(f"Average V/W compression ratio: {np.mean(vw_compression):.2f}x")

## Load Projections for Evaluation

Load computed projections for evaluation.

In [ ]:
# Load different projection methods for comparison
list_proj_svd = read_proj(model, proj_name)
list_proj_eigen = read_proj(model, eigen_proj_name)
list_proj_kqt = read_proj(model, kqt_proj_name)  # Uncomment if KQT projections were computed

print("Projections loaded successfully")
print(f"SVD projections: {len(list_proj_svd)} layers")
print(f"Eigen projections: {len(list_proj_eigen)} layers")
print(f"KQT projections: {len(list_proj_kqt)} layers")

## Evaluation Function for Mistral

Define validation error computation adapted for Mistral's sliding window attention.

In [ ]:
def validation_error_mistral(model, val_sequences, list_proj, ranks, sliding_window=4096):
    """
    Compute validation errors for Mistral model with sliding window attention.
    """
    all_errors = [[] for _ in range(7)]  # 7 different error metrics
    
    for i, seq in enumerate(val_sequences):
        print(f"Processing sequence {i+1}/{len(val_sequences)}")
        
        # Create cache and run forward pass
        cache = CustomCacheMistral(sliding_window=sliding_window)
        with torch.no_grad():
            model(seq.to(device), past_key_values=cache, use_cache=True)
        
        # Convert cache to evaluation format
        past_key_values = []
        for layer_idx in range(len(cache)):
            k, v = cache[layer_idx][:2]  # Get keys and values
            q = cache.get_query_states(layer_idx) if hasattr(cache, 'get_query_states') else k
            past_key_values.append((k, v, q))
        
        # Compute errors using Mistral-specific evaluation
        errors = testing_proj_mistral(model, past_key_values, list_proj, ranks, sliding_window)
        
        for j, error_list in enumerate(errors):
            all_errors[j].extend(error_list)
    
        
    # Average errors across sequences and layers
    averaged_errors = []
    for error_list in all_errors:
        # Convert tensors to CPU and numpy, then reshape and average
        cpu_errors = [err.cpu().numpy() if torch.is_tensor(err) else err for err in error_list]
        error_array = np.array(cpu_errors).reshape(len(val_sequences), -1)
        averaged_errors.append(np.mean(error_array, axis=0))

    
    return averaged_errors

## Run Evaluation

Evaluate different compression methods on validation data.

In [ ]:
# Set evaluation parameters
eval_ranks = ranks

# print(f"Evaluating with rank {eval_rank}")
print(f"Using {len(valdata[:32])} validation sequences")  # Limit for faster evaluation
print(eval_ranks)

In [ ]:
# Evaluate SVD method
print("Evaluating SVD projections...")
svd_errors = validation_error_mistral(
    model, 
    valdata, 
    list_proj_svd, 
    eval_ranks, 
    sliding_window=sliding_window
)

print("SVD evaluation completed")

In [ ]:
# Evaluate Eigen method
print("Evaluating Eigen projections...")
eigen_errors = validation_error_mistral(
    model, 
    valdata, 
    list_proj_eigen, 
    eval_ranks, 
    sliding_window=sliding_window
)

print("Eigen evaluation completed")

In [ ]:
# Evaluate KQT method
print("Evaluating KQT projections...")
kqt_errors = validation_error_mistral(
    model, 
    valdata, 
    list_proj_kqt, 
    eval_ranks, 
    sliding_window=sliding_window
)

print("KQT evaluation completed")

## Results Visualization

Create comprehensive plots comparing different compression methods.

In [ ]:
def plotting_all_three_methods_mistral(svd_errors, eigen_errors, kqt_errors, save=False, save_name=None):
    """
    Plot comprehensive comparison of all three compression methods for Mistral model.
    """
    f, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 12))
    
    # Colors for the three methods
    colors = {'SVD': 'lightblue', 'Eigen': 'firebrick', 'KQT': 'forestgreen'}
    
    # Plot 1: Cache approximation errors
    ax1.plot(range(1, L+1), svd_errors[0], color=colors['SVD'], marker="s", label="SVD: K cache", alpha=0.7)
    ax1.plot(range(1, L+1), svd_errors[2], color=colors['SVD'], marker="o", label="SVD: V cache", alpha=0.7)
    ax1.plot(range(1, L+1), svd_errors[1], color=colors['SVD'], marker="^", label="SVD: Q cache", alpha=0.7)
    
    ax1.plot(range(1, L+1), eigen_errors[0], color=colors['Eigen'], marker="s", label="Eigen: K cache")
    ax1.plot(range(1, L+1), eigen_errors[2], color=colors['Eigen'], marker="o", label="Eigen: V cache")
    ax1.plot(range(1, L+1), eigen_errors[1], color=colors['Eigen'], marker="^", label="Eigen: Q cache")
    
    ax1.plot(range(1, L+1), kqt_errors[0], color=colors['KQT'], marker="s", label="KQT: K cache", linewidth=2)
    ax1.plot(range(1, L+1), kqt_errors[2], color=colors['KQT'], marker="o", label="KQT: V cache", linewidth=2)
    ax1.plot(range(1, L+1), kqt_errors[1], color=colors['KQT'], marker="^", label="KQT: Q cache", linewidth=2)
    
    ax1.set_xlabel("Layer index")
    ax1.set_ylabel("Relative Frobenius norm error")
    ax1.set_title("Mistral: Cache Matrix Approximation Errors")
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Attention matrix errors  
    ax2.plot(range(1, L+1), svd_errors[3], color=colors['SVD'], marker="s", label="SVD", alpha=0.7)
    ax2.plot(range(1, L+1), eigen_errors[3], color=colors['Eigen'], marker="s", label="Eigen")
    ax2.plot(range(1, L+1), kqt_errors[3], color=colors['KQT'], marker="s", label="KQT", linewidth=2)

    print(svd_errors[3],eigen_errors[3],kqt_errors[3])
    
    ax2.set_xlabel("Layer index")
    ax2.set_ylabel("Relative Frobenius norm error")
    ax2.set_title("Mistral: Attention Matrix Approximation Errors")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Final output errors
    ax3.plot(range(1, L+1), svd_errors[6], color=colors['SVD'], marker="s", label="SVD", alpha=0.7)
    ax3.plot(range(1, L+1), eigen_errors[6], color=colors['Eigen'], marker="s", label="Eigen")
    ax3.plot(range(1, L+1), kqt_errors[6], color=colors['KQT'], marker="s", label="KQT", linewidth=2)
    
    ax3.set_xlabel("Layer index")
    ax3.set_ylabel("Relative Frobenius norm error")
    ax3.set_title("Mistral: Attention Output Approximation Errors")
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Comprehensive comparison summary
    metrics = ['K Error', 'Q Error', 'V Error', 'Attn Error', 'Output Error']
    indices = [0, 1, 2, 3, 6]
    
    svd_means = [np.mean(svd_errors[i]) for i in indices]
    eigen_means = [np.mean(eigen_errors[i]) for i in indices]
    kqt_means = [np.mean(kqt_errors[i]) for i in indices]
    
    x = np.arange(len(metrics))
    width = 0.25
    
    ax4.bar(x - width, svd_means, width, label='SVD', color=colors['SVD'], alpha=0.7)
    ax4.bar(x, eigen_means, width, label='Eigen', color=colors['Eigen'], alpha=0.8)
    ax4.bar(x + width, kqt_means, width, label='KQT', color=colors['KQT'], alpha=0.9)
    
    ax4.set_xlabel('Error Type')
    ax4.set_ylabel('Mean Error Across Layers')
    ax4.set_title('Mistral: Average Compression Errors Comparison')
    ax4.set_xticks(x)
    ax4.set_xticklabels(metrics, rotation=45)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_yscale('log')
    
    plt.tight_layout()
    
    if save:
        assert save_name is not None
        f.savefig(save_name, dpi=300, bbox_inches='tight')
    
    plt.show()
    
    # Print comprehensive summary statistics
    print("\n" + "="*70)
    print("COMPLETE MISTRAL COMPRESSION METHODS COMPARISON")
    print("="*70)
    print(f"Model: {type(model).__name__}")
    print(f"Sliding Window: {sliding_window}")
    
    print("\nMean Errors Across All Layers:")
    error_labels = ['K Cache', 'Q Cache', 'V Cache', 'Attn Matrix', 'Attn Coeff', 'V@WO', 'Output']
    
    for i, label in enumerate(error_labels):
        svd_err = np.mean(svd_errors[i])
        eigen_err = np.mean(eigen_errors[i])
        kqt_err = np.mean(kqt_errors[i])
        
        print(f"  {label:12} - SVD: {svd_err:.6f}, Eigen: {eigen_err:.6f}, KQT: {kqt_err:.6f}")
        
        # Find the best method for this metric
        errors = {'SVD': svd_err, 'Eigen': eigen_err, 'KQT': kqt_err}
        best_method = min(errors, key=errors.get)
        improvement_vs_svd = (svd_err - errors[best_method]) / svd_err * 100 if best_method != 'SVD' else 0
        print(f"                     → Best: {best_method} ({improvement_vs_svd:.1f}% better than SVD)")
    
    # Key performance metrics comparison
    print(f"\n" + "="*50)
    print("KEY PERFORMANCE SUMMARY:")
    print("="*50)
    output_errors = {'SVD': np.mean(svd_errors[6]), 'Eigen': np.mean(eigen_errors[6]), 'KQT': np.mean(kqt_errors[6])}
    best_overall = min(output_errors, key=output_errors.get)
    
    print(f"Overall Best Method (Output Error): {best_overall}")
    print(f"  - SVD Output Error:   {output_errors['SVD']:.6f}")
    print(f"  - Eigen Output Error: {output_errors['Eigen']:.6f}")
    print(f"  - KQT Output Error:   {output_errors['KQT']:.6f}")
    
    # Calculate relative improvements
    if best_overall != 'SVD':
        svd_improvement = (output_errors['SVD'] - output_errors[best_overall]) / output_errors['SVD'] * 100
        print(f"  - {best_overall} is {svd_improvement:.1f}% better than SVD")
    
    if best_overall != 'Eigen':
        eigen_improvement = (output_errors['Eigen'] - output_errors[best_overall]) / output_errors['Eigen'] * 100
        print(f"  - {best_overall} is {eigen_improvement:.1f}% better than Eigen")
    
    print("="*70)

In [ ]:
# Generate comprehensive comparison plots for all three methods
plotting_all_three_methods_mistral(
    svd_errors, 
    eigen_errors, 
    kqt_errors,
    save=False, 
    save_name=f"mistral_all_three_methods_comparison.png"
)

In [ ]:
def save_error(error, filename):
    torch.save(torch.tensor(error),filename)

def load_error(filename):
    return torch.load(filename).tolist()

In [ ]:
save_error(kqt_errors,"error_Mistral_7B_KQT_valdata32_2048_C4.pt")
save_error(svd_errors,"error_Mistral_7B_SVD_valdata32_2048_C4.pt")
save_error(eigen_errors,"error_Mistral_7B_SVDEigen_valdata32_2048_C4.pt")